# Mini World Models — V · M · C on a racing track

**Companion notebook for Lecture 3 of *Build a World Model from Scratch* (Vizuara AI).**

In 2018, David Ha and Jürgen Schmidhuber published *World Models* — the paper that gave
this whole field its name. Their claim sounded almost absurd: an agent can learn to play
a game **without playing it**, by first learning an internal model of the game and then
practising *inside its own model* — inside a dream.

In this notebook you will reproduce that result, end to end, on a miniature racing game
built right here in the notebook. Nothing is imported from a library of tricks; every
piece is a few dozen lines of PyTorch you can read.

### The map (the same map as the lecture)

| Section | What you build | What's new for you |
|---|---|---|
| **1 · MiniRacer** | a tiny racing game + a dataset of random play | environments, observations vs hidden state |
| **2 · V — Vision** | a convolutional **variational autoencoder** (frame → 16 numbers) | VAE, the reparameterisation trick, why convolutions |
| **3 · M — Memory** | a **GRU** + **mixture density head** + reward head (predicts the *next* code) | recurrent memory, gates, MDN, learned reward |
| **4 · C — Controller** | a 51-parameter driver trained by **evolution, entirely inside M's dream** | evolution strategies, dream training |
| **5 · The verdict** | drop the dream-trained driver onto the *real* track | transfer from imagination to reality |

**How to run:** `Runtime → Run all`. Total time ≈ 15 minutes on the free Colab CPU
(most of it is V's training). No installs, no GPU needed. Every plot you saw on the
lecture slides is generated live by this notebook — same code, your machine.

In [22]:
class MiniRacer:
    SIZE, ACTIONS = 24, 3
    R, HALF_W, SPEED = 8.0, 2.2, 1.5
    GRASS = np.array([0.16, 0.34, 0.16], np.float32)
    ROAD = np.array([0.44, 0.44, 0.46], np.float32)
    KERB = np.array([0.75, 0.72, 0.62], np.float32)
    CAR = np.array([0.92, 0.20, 0.15], np.float32)

    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
        c = (self.SIZE - 1) / 2
        yy, xx = np.mgrid[0:self.SIZE, 0:self.SIZE]
        rad = np.sqrt((xx - c) ** 2 + (yy - c) ** 2)
        self.track = np.tile(self.GRASS, (self.SIZE, self.SIZE, 1)).astype(np.float32)
        self.track[np.abs(rad - self.R) <= self.HALF_W] = self.ROAD
        self.track[np.abs(np.abs(rad - self.R) - self.HALF_W) <= 0.45] = self.KERB
        self._c = c
        self.reset()

    def reset(self):
        self.theta = self.rng.uniform(0, 2 * np.pi)
        self.d = self.rng.uniform(-1.0, 1.0)
        self.phi = self.rng.uniform(-0.2, 0.2)
        self.direction = int(self.rng.choice([-1, 1]))   # the hidden coin flip
        self.t = 0
        return self.observe()

    @property
    def state(self):
        return np.array([self.theta, self.d, self.phi, self.direction], np.float32)

    def observe(self):
        img = self.track.copy()
        r = self.R + self.d
        x = self._c + r * np.cos(self.theta)
        y = self._c + r * np.sin(self.theta)
        xi, yi = int(round(x)), int(round(y))
        for dx in (0, 1):
            for dy in (0, 1):
                px, py = xi + dx - 1, yi + dy - 1
                if 0 <= px < self.SIZE and 0 <= py < self.SIZE:
                    img[py, px] = self.CAR
        return img

    def step(self, action):
        self.phi = float(np.clip(self.phi + 0.16 * (action - 1), -0.7, 0.7))
        self.d = float(self.d + 1.1 * np.sin(self.phi))
        if abs(self.d) > self.HALF_W:                    # scraping the kerb
            self.d = float(np.clip(self.d, -self.HALF_W, self.HALF_W))
            self.phi *= 0.4
        self.theta = (self.theta + self.direction * self.SPEED / (self.R + self.d)) % (2 * np.pi)
        reward = 1.0 - abs(self.d) / self.HALF_W
        self.t += 1
        return self.observe(), float(reward), self.t >= 150

## 0 · Setup

Standard imports, a fixed random seed (so your numbers match the lecture's), and the
Vizuara plot style. Nothing conceptual here — run it and move on.

## 1 · The world: MiniRacer

Before any neural network, we need a **world** — in reinforcement-learning language, an
*environment*. An environment is a little machine with three moves:

* `reset()` → puts the world in a fresh starting state and hands you the first
  **observation** (here: a 24×24×3 RGB image — 1,728 numbers),
* `step(action)` → advances the world one tick and returns the new observation, a
  **reward** (one number telling you how well that tick went), and a done flag,
* an **episode** = one run from `reset()` until done (here: 150 steps).

MiniRacer is a ring road on grass. The car moves at constant speed; the agent only
steers: action `0` = steer left, `1` = straight, `2` = steer right. The reward each
step is `1 − |distance from the road's centre-line| / half-width` — full marks for
driving dead-centre, zero for scraping the kerb. Our stand-in for the paper's CarRacing.

**The crucial design choice — read this twice.** At every `reset()`, a hidden coin flip
decides whether the car circulates **clockwise or counter-clockwise** for the whole
episode. That direction is *real state of the world*, but it is **invisible in any single
frame**: a frame shows *where* the car is, never *which way it's going*. This is Lecture 1's
state-vs-observation gap, built into the game on purpose — later, it is exactly what
forces M to carry memory and to predict a *menu* of futures rather than one guess.

In [23]:
import math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import rcParams

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- Vizuara warm-paper plot style ------------------------------------------
PAPER, INK, MUTED = "#FBF9F1", "#16130D", "#6D665A"
TEAL, GOLD, CLAY = "#2E8F8F", "#DD9F3E", "#C96442"
rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.family": "serif", "axes.titlesize": 13, "axes.titleweight": "bold",
})

device: cpu


### 1.1 · Collect a dataset of random play

A world model is learned **from experience**, so first we need experience. We let a
completely **random driver** play 150 episodes × 150 steps = **22,500 frames**, and we
record three things at every step: the frame, the action taken, and the reward
received — and, crucially, we keep them **in sequence** (episode by episode, step by
step), because M will need to learn how one step follows another.

Why a *random* driver? Because this is the only real experience the whole system will
ever get. V, M and C are all built from this one clumsy, aimless dataset — that is
part of what makes the final result surprising. (Why record the reward? Hold that
thought until Section 3 — M will need to learn it.)

In [ ]:
# collect experience with a random policy — sequences, because M needs time.
env = MiniRacer(seed=SEED)
EPISODES, T = 150, 150
frames = np.zeros((EPISODES, T, 24, 24, 3), np.float32)
actions = np.zeros((EPISODES, T), np.int64)
rewards = np.zeros((EPISODES, T), np.float32)
rng = np.random.default_rng(SEED)
for e in range(EPISODES):
    obs = env.reset()
    for t in range(T):
        a = int(rng.integers(0, 3))
        frames[e, t], actions[e, t] = obs, a
        obs, r, done = env.step(a)
        rewards[e, t] = r
print(f"dataset: {EPISODES} episodes × {T} steps = {EPISODES*T:,} frames "
      f"({frames.nbytes/1e6:.0f} MB)")

fig, axes = plt.subplots(1, 8, figsize=(12, 1.9))
for i in range(8):
    axes[i].imshow(frames[i, i * 9]); axes[i].axis("off")
fig.suptitle("MiniRacer — eight raw observations from the dataset", y=1.12)
plt.tight_layout(); plt.savefig("plot_env_frames.png", dpi=150, bbox_inches="tight"); plt.show()

## 2 · V — the Vision model: a convolutional variational autoencoder

**The job:** squeeze each frame (24×24×3 = **1,728 numbers**) down to a code **z of
just 16 numbers**, such that the frame can be redrawn from the code. Sixteen numbers
is plenty here, because the only thing that ever changes in this world is *where the
car is* — the track is wallpaper.

**New concept — the autoencoder.** Two networks trained together: an **encoder**
(frame → code) and a **decoder** (code → frame). The training signal is simply
*reconstruction error* — how different is the redrawn frame from the original? To do
well, the encoder is forced to keep exactly the information that matters. Compression
by necessity, not by instruction.

**New concept — the *variational* autoencoder (VAE).** A plain autoencoder maps each
frame to a single *point* in code space — and the space *between* points it has seen
can decode to garbage. The VAE's fix: encode each frame to a small **region** instead —
a centre `μ` and a spread `σ` (16 numbers each) — and during training, decode a point
*sampled from that region*: `z = μ + σ·noise`. Because the decoder must handle
everywhere inside every region, the space between codes becomes smooth and meaningful.
Two details you're seeing for the first time:

* **The reparameterisation trick.** We can't backpropagate through "draw a random
  sample" — so we rewrite the sample as `deterministic(μ, σ) + noise`: the randomness
  is pulled out into an external `noise` that needs no gradient, and gradients flow
  cleanly through `μ` and `σ`. One line of code, and it is what makes VAEs trainable.
* **The KL term** in the loss gently pushes every region toward a standard bell curve
  (mean 0, spread 1) — it stops the encoder from "cheating" by shrinking `σ` to zero
  and quietly becoming a plain autoencoder again.

Why do we care so much about a *smooth* code space? Because in Section 3, **M will
predict codes** — and its predictions will land *near* real codes, never exactly on
them. In a smooth space, "near a code" still decodes to a sane frame. In an island
space, the dream shatters on step one.

**New concept — convolutional layers (and why we must use them here).** A dense layer
connects every input pixel to every neuron; a **conv layer** slides one small filter
(here 4×4) across the whole image, applying the *same weights everywhere*. That weight
sharing means a "red blob detector" learned in one corner works in every corner.
We learned the hard way that this is not optional: the car is **4 pixels out of 576**
(0.7% of the input), and a fully-connected V drowns that signal in the constant
background — its loss flat-lines and its reconstructions show a *perfect empty track
with no car*. This is exactly why Ha & Schmidhuber's V is convolutional.

Two more honest engineering details, straight from the trenches:

* **We weight the pixels that move.** The car is 12 of 1,728 numbers — to a plain
  reconstruction loss, ignoring it entirely costs almost nothing (painting the empty
  track already scores 99%). So pixels that differ from the empty track get ~30×
  weight: *reconstruct what changes, not what's wallpaper.*
* **We add a little extra fuzz to z during training** (`+0.10·noise`) so the decoder
  is trained on a slightly wider neighbourhood of each code — insurance for exactly
  the near-miss codes M will produce later.

Watch the training printout: the loss sits on a **plateau** for a few epochs (V paints
the empty track, no car) and then breaks through once the conv filters lock onto the
red blob. Don't panic at the flat start — that plateau is real, and it's a preview of
how optimisation often *feels* on small-signal problems.

In [24]:
Z = 16

class ConvVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(                            # in: 3 × 24 × 24
            nn.Conv2d(3, 16, 4, stride=2, padding=1), nn.ReLU(),   # 16 × 12 × 12
            nn.Conv2d(16, 32, 4, stride=2, padding=1), nn.ReLU(),  # 32 × 6 × 6
            nn.Flatten())                                    # 1152
        self.mu, self.logvar = nn.Linear(1152, Z), nn.Linear(1152, Z)
        self.fc = nn.Linear(Z, 1152)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 4, stride=2, padding=1), nn.Sigmoid())

    def encode(self, x):                                     # x: (B, 24, 24, 3)
        h = self.enc(x.permute(0, 3, 1, 2))
        return self.mu(h), self.logvar(h)

    def decode(self, z):                                     # -> (B, 24, 24, 3)
        return self.dec(self.fc(z).view(-1, 32, 6, 6)).permute(0, 2, 3, 1)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)   # reparameterise
        return self.decode(z), mu, logvar

vae = ConvVAE().to(device)
opt = torch.optim.Adam(vae.parameters(), lr=2e-3)
flat = torch.tensor(frames.reshape(-1, 24, 24, 3))
track_t = torch.tensor(env.track).to(device)                 # the empty track
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(flat),
                                     batch_size=256, shuffle=True)
t0 = time.time()
for epoch in range(30):
    tot = 0.0
    for (x,) in loader:
        x = x.to(device)
        mu, logvar = vae.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)   # reparameterise
        z = z + 0.10 * torch.randn_like(z)   # extra fuzz: M's predictions will land
                                             # NEAR codes, so train the decoder there too
        xhat = vae.decode(z)
        w = 1.0 + 30.0 * (x - track_t).abs().mean(-1, keepdim=True)  # weight what moves
        recon = (w * F.binary_cross_entropy(xhat, x, reduction="none")).sum() / len(x)
        kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu**2 - logvar.exp(), dim=1))
        loss = recon + 1.0 * kl
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(x)
    if (epoch + 1) % 3 == 0:
        print(f"epoch {epoch+1:2d}  loss {tot/len(flat):8.2f}")
print(f"V trained in {time.time()-t0:.0f}s   "
      f"({sum(p.numel() for p in vae.parameters()):,} parameters)")
# note: the loss sits on a plateau for a few epochs (V paints the empty track,
# no car) and then breaks through once the conv filters lock onto the red blob.

AttributeError: module 'torch' has no attribute '_utils'

### V's report card: reconstructions and the smooth walk

Two tests, matching the two claims we made about V.

**Test 1 — can V redraw a frame from 16 numbers?** Top row: real frames. Bottom row:
what the decoder paints back from *only the code*. Look for the car being repainted in
the right place — that's the entire point of the weighted loss.

**Test 2 — is the space *between* codes smooth?** We take two real frames from the
*same episode*, 26 steps apart (the car has driven a third of a lap), encode both to
get **code A** and **code B**, then build seven **invented codes** by sliding from A to
B (`0.875·A + 0.125·B`, `0.75·A + 0.25·B`, …). **No frame ever produced these codes.**
If the VAE did its job, each one still decodes to a sane track with the car somewhere
*between* A's position and B's — the car should slide along the ring. (It dims a little
mid-walk, where no training frame ever lived — that's honest, and good enough for M.)

In [ ]:
vae.eval()
with torch.no_grad():
    test = flat[:20000:2611][:8].to(device)
    recon, _, _ = vae(test)

    fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
    for i in range(8):
        axes[0, i].imshow(test[i].cpu()); axes[0, i].axis("off")
        axes[1, i].imshow(recon[i].cpu().clamp(0, 1)); axes[1, i].axis("off")
    axes[0, 0].set_title("original", loc="left", color=TEAL)
    axes[1, 0].set_title("V's reconstruction (from 16 numbers)", loc="left", color=CLAY)
    fig.suptitle("V — 1,728 pixels → 16 numbers → 1,728 pixels", y=1.02)
    plt.tight_layout(); plt.savefig("plot_v_recon.png", dpi=150, bbox_inches="tight"); plt.show()

    # latent walk between two frames of the SAME episode, half a lap apart
    e_walk = 2
    z1, _ = vae.encode(torch.tensor(frames[e_walk, 0:1]).to(device))
    z2, _ = vae.encode(torch.tensor(frames[e_walk, 26:27]).to(device))
    steps = torch.linspace(0, 1, 9, device=device).view(-1, 1)
    walk = vae.decode(z1 + steps * (z2 - z1))

    fig, axes = plt.subplots(1, 9, figsize=(12, 1.8))
    for i in range(9):
        axes[i].imshow(walk[i].cpu().clamp(0, 1)); axes[i].axis("off")
    axes[0].set_title("code A", loc="left", color=TEAL)
    axes[-1].set_title("code B", loc="right", color=GOLD)
    fig.suptitle("The smooth walk — the car glides around the track between two codes", y=1.15)
    plt.tight_layout(); plt.savefig("plot_v_walk.png", dpi=150, bbox_inches="tight"); plt.show()

NameError: name 'vae' is not defined

## 3 · M — the Memory model: a GRU with a mixture head (and a reward head)

**The job:** given the current code, the action taken, and everything seen so far,
predict the **next code**: `(z_now, action, memory) → ẑ_next`. This is the heart of
the paper — the network that *imagines*. Three new ideas live inside it.

**New concept — recurrent memory (the GRU).** One MiniRacer frame cannot tell you the
direction of travel — so no network that sees only the current frame can predict the
next one. M therefore carries a **hidden state `h`** (here, 160 numbers): think of it
as an index card the network *rewrites at every step*, blending three ingredients —
the old card, the new code `z`, and the action `a`. The card is never wiped during an
episode, so a fact learned at step 2 ("this episode runs clockwise" — discovered by
comparing two consecutive positions) can still be on the card at step 149. The **GRU**
(gated recurrent unit) is the specific cell we use: it has a learned **update gate**
`u ∈ [0,1]` per card entry, and updates each entry as
`h_new = (1−u)·h_old + u·h̃`. Gate shut (`u≈0`) → the entry is *copied through
unchanged* — that pure-copy path is the entire secret of long memory. Gate open
(`u≈1`) → the entry is rewritten with fresh evidence. (The paper uses the LSTM —
same protection idea, one more gate.)

**New concept — the mixture density network (MDN).** Here the hidden coin flip bites.
Early in an episode, the *true* next position is genuinely two-sided: clockwise or
counter-clockwise, both really happen in the data. A network trained to output ONE
guess learns the *average* of the two — a car that barely moves, a future that never
happens even once. The MDN's fix: output a **menu of K = 5 candidate futures**, each
with a probability — three lists read off the card by three small linear layers:

* `π` — 5 branch probabilities (which future?),
* `μ_k` — the centre of branch k (a full 16-dim code),
* `σ_k` — how fuzzy branch k is.

To *predict*, roll two dice: pick a branch `k ~ π`, then a point inside it
`ẑ = μ_k + σ_k·noise`. To *train*, compute the probability the **whole menu** assigned
to the code that actually came next, and push it up (that's the `logsumexp` in
`mdn_loss` — the log of a sum of per-branch probabilities). A menu that kept both
sides of a real fork alive scores well; a menu that averaged them scores terribly —
so honest uncertainty is literally what the loss pays for.

**New concept — a learned reward head.** M also gets a fourth, tiny head:
`r̂(h)` — one number, "how well is this step scoring?" Why: in Section 4, C will train
*inside M's dream*, and a dream has no game engine to hand out scores. If the dream is
to be a complete substitute for the game, **M must learn the reward too** (that is why
we recorded rewards back in Section 1.1). The paper's M predicts "done" for Doom the
same way; every modern world model — Dreamer included — has exactly this head.

One more paper-faithful detail: we train M on codes **sampled** from V's regions
(`μ + σ·noise`), not on the clean centres — so M learns to be robust to exactly the
kind of near-miss codes it will later produce itself.

In [ ]:
K, H = 5, 160

class MDNRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.GRU(Z + 3, H, batch_first=True)
        self.pi = nn.Linear(H, K)
        self.mu = nn.Linear(H, K * Z)
        self.logsig = nn.Linear(H, K * Z)
        self.rhead = nn.Linear(H, 1)          # predicts this step's reward

    def forward(self, z, a_onehot, h=None):
        out, h = self.rnn(torch.cat([z, a_onehot], -1), h)
        B, T_, _ = out.shape
        return (self.pi(out), self.mu(out).view(B, T_, K, Z),
                self.logsig(out).view(B, T_, K, Z).clamp(-5, 2),
                self.rhead(out).squeeze(-1), h)

def mdn_loss(pi, mu, logsig, target):
    t = target.unsqueeze(2)
    logp = -0.5 * (((t - mu) / logsig.exp()) ** 2 + 2 * logsig + math.log(2 * math.pi))
    logp = logp.sum(-1) + F.log_softmax(pi, -1)
    return -torch.logsumexp(logp, -1).mean()

with torch.no_grad():
    zs_parts = []
    for i in range(0, len(flat), 4096):
        m, lv = vae.encode(flat[i:i+4096].to(device))
        zs_parts.append((m + torch.randn_like(m) * (0.5 * lv).exp()).cpu())  # sampled!
    zs = torch.cat(zs_parts).view(EPISODES, T, Z)
a1h = F.one_hot(torch.tensor(actions), 3).float()
rew = torch.tensor(rewards)

mdn = MDNRNN().to(device)
opt = torch.optim.Adam(mdn.parameters(), lr=1e-3)
t0 = time.time()
for epoch in range(30):
    perm, tot = torch.randperm(EPISODES), 0.0
    for i in range(0, EPISODES, 16):
        idx = perm[i:i+16]
        z = zs[idx].to(device); a = a1h[idx].to(device); r = rew[idx].to(device)
        pi, mu, logsig, rhat, _ = mdn(z[:, :-1], a[:, :-1])
        loss = mdn_loss(pi, mu, logsig, z[:, 1:]) + 10.0 * F.mse_loss(rhat, r[:, :-1])
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(idx)
    if (epoch + 1) % 3 == 0:
        print(f"epoch {epoch+1:2d}  loss {tot/EPISODES:8.3f}")
print(f"M trained in {time.time()-t0:.0f}s   "
      f"({sum(p.numel() for p in mdn.parameters()):,} parameters)")

NameError: name 'rng' is not defined

### Dreaming — and watching the dream drift

**New concept — the closed loop (a "dream").** So far M has only ever predicted one
step ahead from *real* data. Now we do the audacious thing: **feed M's own sampled
prediction back in as if it were real**, and predict again, and again. M is now
generating an endless imagined episode with no game engine anywhere — the field calls
this a *rollout in imagination*, or simply a dream. (V's decoder is only used so *we*
can watch; the loop itself runs entirely in 16-number code space.)

The `temperature` knob in `sample_mdn` scales how adventurously we sample from the
menu (τ < 1 = calmer than the data, τ > 1 = wilder). Remember it — it becomes the
cure for a disease we'll meet in Section 4.

**The experiment below:** warm M up on just 4 real frames, unplug reality, and dream
30 steps forward using the same actions the real episode took. Top row: what the real
track actually did. Bottom row: M's dream. Then we quantify the divergence over 30
different dreams — watch the error **climb for ~10 steps and then plateau**: the dream
departs from *the truth* but settles into *a* plausible lap of its own. This is
**compounding error** — every world model ever built fights this curve.

In [ ]:
def sample_mdn(pi, mu, logsig, temperature=1.0):
    probs = F.softmax(pi / temperature, -1)
    k = torch.multinomial(probs, 1).squeeze(-1)
    sel_mu = mu[torch.arange(len(k)), k]
    sel_sig = logsig[torch.arange(len(k)), k].exp() * math.sqrt(temperature)
    return sel_mu + torch.randn_like(sel_mu) * sel_sig

mdn.eval()
DREAM = 30

def dream_rollout(e, temperature=0.7):
    z_seq = zs[e:e+1].to(device); a_seq = a1h[e:e+1].to(device)
    with torch.no_grad():
        _, _, _, _, h = mdn(z_seq[:, :4], a_seq[:, :4], None)
        z, dream_z = z_seq[:, 4:5], []
        for t in range(4, 4 + DREAM):
            pi, mu, logsig, _, h = mdn(z, a_seq[:, t:t+1], h)
            z = sample_mdn(pi[:, 0], mu[:, 0], logsig[:, 0], temperature).unsqueeze(1)
            dream_z.append(z.squeeze(1))
        return vae.decode(torch.cat(dream_z)).cpu()

with torch.no_grad():
    e = 3
    dream_frames = dream_rollout(e)
    real_frames = frames[e, 5:5 + DREAM]
    all_d = []
    for ee in range(30):
        df = dream_rollout(ee).numpy()
        all_d.append(np.sqrt(((df - frames[ee, 5:5 + DREAM]) ** 2).sum((1, 2, 3))))
    drift = np.mean(all_d, 0)

show = [0, 4, 9, 14, 19, 24, 29]
fig, axes = plt.subplots(2, len(show), figsize=(12, 3.6))
for i, t in enumerate(show):
    axes[0, i].imshow(real_frames[t]); axes[0, i].axis("off")
    axes[0, i].set_title(f"t+{t+1}", fontsize=9, color=MUTED)
    axes[1, i].imshow(dream_frames[t].clamp(0, 1)); axes[1, i].axis("off")
axes[0, 0].set_title("real", loc="left", color=TEAL)
axes[1, 0].set_title("M's dream", loc="left", color=CLAY)
fig.suptitle("The dream vs reality — same start, same actions", y=1.02)
plt.tight_layout(); plt.savefig("plot_m_dream.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(7.5, 3.4))
lo = np.percentile(all_d, 25, 0); hi = np.percentile(all_d, 75, 0)
ax.plot(range(1, DREAM + 1), drift, color=CLAY, lw=2.5)
ax.fill_between(range(1, DREAM + 1), lo, hi, color=CLAY, alpha=0.15)
ax.set_xlabel("dream steps"); ax.set_ylabel("pixel error vs reality")
ax.set_title("Compounding error: the dream departs from the truth, then stays gone")
plt.tight_layout(); plt.savefig("plot_m_drift.png", dpi=150, bbox_inches="tight"); plt.show()

## 4 · C — the Controller: 51 parameters, trained inside the dream

**The job:** read the code `z` and output an action. C is deliberately, almost
insultingly small — **one linear layer**: `z (16) → 3 steering scores`, take the
biggest. That's 16×3 weights + 3 biases = **51 parameters**. The philosophy (the
paper's philosophy): put all the intelligence into *understanding* the world (V and M
together hold 188,073 parameters); if the situation is well-summarised, *acting* on it
is nearly trivial. Small C is the proof that the world model earned its keep.

**New concept — evolution strategies (training without gradients).** With only 51
numbers, we don't need backpropagation at all. Each generation we: ① make 12 random
tweak directions and try each one both ways (**mirrored sampling**: θ+ε and θ−ε — a
free variance reducer, since one of the pair is usually better), ② score all 24
candidates, ③ nudge θ toward the tweaks that *ranked* best (**rank-based** updates
ignore the raw scores, so one lucky dream can't dominate). One more trick:
every candidate in a generation is scored on the **same dream seeds** (same starting
episodes, same dice) — a fair race, so score differences reflect the *policy*, not luck.

**And now the experiment the field is named after.** Every single episode C ever
drives during training is **M's dream** — and every score it receives is **M's own
imagined reward `r̂`**. The real game is never touched during learning. After each
generation we *measure* (never train!) on the real track, just to watch.

*(Confession from building this notebook: our first attempt scored dreams by decoding
the frames and measuring the red car's position with pixel arithmetic. C learned to
**exploit the scoring** — dream scores held steady while real driving got worse.
"The policy cheats the model's flaws" is the oldest disease of world models — the
paper hit it in dream-Doom, we hit it here. Switching the score to M's learned reward
head fixed it. Exercise ⑤ lets you re-create the disaster.)*

In [ ]:
def dream_return(params, seeds, horizon=40, temperature=1.0):
    """Drive C inside M's dream. `seeds` fixes the dream episodes and dice for
    the whole population — every candidate faces the SAME dreams (fair race)."""
    W = torch.tensor(params[:48], dtype=torch.float32, device=device).view(3, Z)
    b = torch.tensor(params[48:], dtype=torch.float32, device=device)
    total = 0.0
    with torch.no_grad():
        for (e, ts) in seeds:
            g = torch.Generator().manual_seed(ts)
            _, _, _, _, h = mdn(zs[e:e+1, :4].to(device), a1h[e:e+1, :4].to(device))
            z = zs[e:e+1, 4:5].to(device)
            for t in range(horizon):
                a = int(torch.argmax(W @ z[0, 0] + b))
                ah = F.one_hot(torch.tensor([[a]]), 3).float().to(device)
                pi, mu, logsig, rhat, h = mdn(z, ah, h)
                total += float(rhat[0, 0].clamp(0, 1))      # M's own score
                probs = F.softmax(pi[0, 0] / temperature, -1)
                k = int(torch.multinomial(probs, 1, generator=g))
                sig = logsig[0, 0, k].exp() * math.sqrt(temperature)
                z = (mu[0, 0, k] + torch.randn(Z, generator=g).to(device) * sig).view(1, 1, Z)
    return total / len(seeds)

def real_return(params, n_eps=10, horizon=100):
    W = torch.tensor(params[:48], dtype=torch.float32, device=device).view(3, Z)
    b = torch.tensor(params[48:], dtype=torch.float32, device=device)
    scores = []
    with torch.no_grad():
        for k in range(n_eps):
            env2 = MiniRacer(seed=1000 + k)
            obs, tot = env2.reset(), 0.0
            for t in range(horizon):
                zt, _ = vae.encode(torch.tensor(obs).view(1, 24, 24, 3).to(device))
                a = int(torch.argmax(W @ zt[0] + b))
                obs, r, done = env2.step(a)
                tot += r
            scores.append(tot)
    return float(np.mean(scores)), float(np.std(scores))

# evolution: mirrored perturbation pairs + rank-based update
NPAIR, GENS, SIGMA, LR = 12, 20, 0.5, 0.25
theta = np.zeros(51)
hist_dream, hist_real, t0 = [], [], time.time()
base_real, _ = real_return(theta)
seeds0 = [(int(rng.integers(0, EPISODES)), int(rng.integers(0, 10**6))) for _ in range(6)]
hist_dream.append(dream_return(theta, seeds0))
hist_real.append(base_real)
for gnum in range(GENS):
    seeds = [(int(rng.integers(0, EPISODES)), int(rng.integers(0, 10**6))) for _ in range(6)]
    half = np.random.randn(NPAIR, 51) * SIGMA
    noise = np.concatenate([half, -half])                   # mirrored: tweak and anti-tweak
    scores = np.array([dream_return(theta + n, seeds) for n in noise])
    ranks = (scores.argsort().argsort() - (len(noise) - 1) / 2) / len(noise)
    theta = theta + LR * (ranks @ noise) / SIGMA
    hist_dream.append(dream_return(theta, seeds))
    hist_real.append(real_return(theta, n_eps=6)[0])   # measured, never trained on
    print(f"gen {gnum+1:2d}  dream {hist_dream[-1]:6.2f}   real {hist_real[-1]:6.2f}")
print(f"C evolved in {time.time()-t0:.0f}s")

trained_real, trained_std = real_return(theta)
rand_scores = [real_return(np.random.randn(51) * 0.5, n_eps=3)[0] for _ in range(4)]
print(f"\nreal-track return (100 steps)  |  untrained: {base_real:.1f}   "
      f"random: {np.mean(rand_scores):.1f}   "
      f"dream-trained: {trained_real:.1f} ± {trained_std:.1f}")

The verdict: dream driving, real skill
Read the left panel carefully — it is the whole paper in one plot. The gold curve is C's score inside the dream (the only thing it ever trained on). The teal curve is its score on the real track, which we only ever measured. The teal curve climbs anyway: skill learned in imagination is showing up in reality.

The bars on the right put numbers on it. untrained = θ = 0 (it steers hard left forever and pins itself to the kerb). random controllers = the average of freshly random 51-parameter policies. trained in the dream = our C. Your exact numbers will match the lecture's (same seed): 2.4 · 27.9 · 47.7 ± 6.4 out of 100.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.5, 1]})
axes[0].plot(range(0, GENS + 1), hist_dream, color=GOLD, lw=2, marker="o", ms=3.5, label="score inside the dream")
axes[0].plot(range(0, GENS + 1), hist_real, color=TEAL, lw=2.5, marker="o", ms=4, label="score on the REAL track")
axes[0].legend(frameon=False, loc="lower right")
axes[0].set_xlabel("generation"); axes[0].set_ylabel("return")
axes[0].set_title("Practice happens in the dream — skill shows up on the track")

bars = axes[1].bar(["untrained", "random\ncontrollers", "trained in\nthe dream"],
                   [base_real, float(np.mean(rand_scores)), trained_real],
                   color=[MUTED, GOLD, TEAL], width=0.62)
axes[1].errorbar(2, trained_real, yerr=trained_std, color=INK, capsize=5, lw=1.5)
axes[1].set_ylabel("return on the REAL track")
axes[1].set_title("…and it transfers")
for b in bars:
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 1.2,
                 f"{b.get_height():.1f}", ha="center", color=INK, fontsize=11, fontweight="bold")
plt.tight_layout(); plt.savefig("plot_c_verdict.png", dpi=150, bbox_inches="tight"); plt.show()

5 · What just happened — and what you now know
V learned to see: 1,728 pixels → 16 numbers → 1,728 pixels, with a smooth space between codes. You met the autoencoder, the variational twist (regions, the reparameterisation trick, the KL term), and convolutions — plus two trench lessons: weight what changes, and train the decoder near the manifold.
M learned to imagine — and to score: a GRU whose gated card carries the hidden direction across 150 steps, a mixture density head that keeps both sides of a genuine fork alive instead of averaging them, and a reward head so the dream can grade you. You closed the loop and watched compounding error — the dream drifting from the truth into a plausible lap of its own.
C — 51 parameters trained by evolution strategies (mirrored, rank-based, common random seeds) — never drove the real track during training. It practised only inside M's dream, graded only by M's imagined rewards… and its skill transferred: 2.4 → 47.7 out of 100.
That is Ha & Schmidhuber (2018) — the first world model — in miniature, on your machine, from a single dataset of random play.

Exercises (each one breaks something instructive)
Turn up the heat. Raise temperature in dream_return to 1.5 — does transfer improve or break? Then lower it to 0.3 — why can an easier dream make a worse real driver? (Hint: what quirks can C rely on in a too-calm dream?)
Longer dreams. In the drift experiment, raise DREAM from 30 to 100 — at what horizon does the dreamed car leave the road entirely? What does that say about how long a dream you can safely train in?
Starve the code. Set Z = 4 and rerun Section 2 — what does V lose first, the track or the car? Why?
Kill the menu. Set K = 1 (a single gaussian — no mixture). Rerun M and dream from 1 warm-up frame, while the clockwise/counter-clockwise coin flip is still secret. Watch what the "average future" does to the dreamed car.
Re-create our cheating disaster. Replace the reward head in dream_return with pixel arithmetic on the decoded frames (find the red car's pixels, measure its distance from radius 8) — then watch dream scores hold steady while real driving collapses. You will have reproduced the oldest failure mode of world models, and you'll never forget it